<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-12/AI_Modul_12.7_Praktikum_YOLO.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 12.7: Praktikum YOLO (Grid Decoding dan Inference Head)
**Tujuan Pembelajaran:**
1. Membangun modul YOLO Detection Head di PyTorch.
2. Mengimplementasikan fungsi dekoding koordinat kisi non-linear ($t_x, t_y, t_w, t_h \to x_1, y_1, x_2, y_2$).
3. Mensimulasikan pipeline inferensi deteksi tandan kelapa sawit pada resolusi spasial $13 \times 13$.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches

torch.manual_seed(42)
print(f"PyTorch Versi: {torch.__version__}")

### Bagian 1: Definisi Modul Mini YOLO Detection Head

In [ ]:
class MiniYOLOHead(nn.Module):
    def __init__(self, in_channels=64, num_anchors=3, num_classes=2):
        super().__init__()
        self.num_anchors = num_anchors
        self.num_classes = num_classes
        # (tx, ty, tw, th, conf) + num_classes
        self.out_dim = num_anchors * (5 + num_classes)
        self.conv = nn.Conv2d(in_channels, self.out_dim, kernel_size=1)
        
    def forward(self, x):
        out = self.conv(x)
        B, C, H, W = out.shape
        out = out.view(B, self.num_anchors, 5 + self.num_classes, H, W)
        return out.permute(0, 1, 3, 4, 2).contiguous()

head = MiniYOLOHead(in_channels=64, num_anchors=3, num_classes=2)
dummy_feat = torch.randn(1, 64, 13, 13)
raw_preds = head(dummy_feat)
print(f"Bentuk Tensor Fitur Input: {dummy_feat.shape}")
print(f"Bentuk Tensor Prediksi YOLO Head: {raw_preds.shape} (Batch, Anchors, H, W, Attrs)")

### Bagian 2: Fungsi Dekoding Tensor Koordinat Menjadi Bounding Box

In [ ]:
def decode_yolo_grid(preds, anchors, img_size=416):
    B, num_anchors, H, W, _ = preds.shape
    device = preds.device
    
    grid_y, grid_x = torch.meshgrid(torch.arange(H, device=device), torch.arange(W, device=device), indexing='ij')
    grid_xy = torch.stack([grid_x, grid_y], dim=-1).view(1, 1, H, W, 2).float()
    anchor_wh = anchors.view(1, num_anchors, 1, 1, 2).to(device)
    
    # Koordinat pusat & ukuran
    stride = img_size / W
    box_xy = (torch.sigmoid(preds[..., 0:2]) + grid_xy) * stride
    box_wh = torch.exp(preds[..., 2:4]) * anchor_wh
    
    x1y1 = box_xy - box_wh / 2.0
    x2y2 = box_xy + box_wh / 2.0
    conf = torch.sigmoid(preds[..., 4:5])
    cls_prob = torch.softmax(preds[..., 5:], dim=-1)
    
    return torch.cat([x1y1, x2y2, conf, cls_prob], dim=-1)

# Inisialisasi anchor boxes [w, h] dalam skala piksel
anchors = torch.tensor([[30., 40.], [60., 70.], [110., 130.]])
decoded_boxes = decode_yolo_grid(raw_preds, anchors, img_size=416)
print(f"Tensor Hasil Dekoding: {decoded_boxes.shape}")
print("[VALIDASI SUKSES] Dekoding koordinat grid YOLO selesai tanpa galat!")

### Bagian 3: Simulasi Deteksi Tandan Buah Sawit dan Visualisasi

In [ ]:
# Simulasi nilai deteksi kuat pada sel kisi (6, 6) anchor 1
mock_preds = torch.zeros(1, 3, 13, 13, 7)
mock_preds[0, 1, 6, 6, 0:2] = 0.5  # Tengah sel
mock_preds[0, 1, 6, 6, 2:4] = 0.0  # Ukuran sama persis anchor
mock_preds[0, 1, 6, 6, 4] = 6.0    # Skor keyakinan sangat tinggi (sigmoid -> ~0.997)
mock_preds[0, 1, 6, 6, 5] = 4.0    # Kelas 0: Tandan Matang

res = decode_yolo_grid(mock_preds, anchors, img_size=416)
target_det = res[0, 1, 6, 6].detach().numpy()

x1, y1, x2, y2, c, p0, p1 = target_det
print(f"Hasil Deteksi Simulasi:")
print(f"  Koordinat BBox: [{x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f}]")
print(f"  Confidence: {c:.4f}")
print(f"  Probabilitas Kelas 0 (Tandan Matang): {p0:.4f}")

# Visualisasi Citra Simulasi
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, 416)
ax.set_ylim(416, 0) # Format citra

# Gambar grid 13x13
for g in range(14):
    pos = g * (416 / 13)
    ax.axvline(pos, color='gray', linestyle=':', alpha=0.4)
    ax.axhline(pos, color='gray', linestyle=':', alpha=0.4)
    
# Bounding box deteksi
rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2.5, edgecolor='#E65100', facecolor='#FFE0B2', alpha=0.6)
ax.add_patch(rect)
ax.text(x1, y1-8, f"Tandan Matang: {c:.2f}", fontsize=10, fontweight='bold', color='#E65100')
plt.title('Simulasi Inferensi Kepala Deteksi YOLO pada Grid 13x13')
plt.tight_layout()
plt.savefig('praktikum_yolo_grid_inference_12_7.png', dpi=200)
plt.close()
print("[OK] Visualisasi inferensi YOLO disimpan sebagai 'praktikum_yolo_grid_inference_12_7.png'")